# 02. Entrenamiento con MLflow

Este notebook entrena el pipeline de LightGBM, registra el experimento en MLflow y muestra los mejores runs del experimento.

In [ ]:
import sys
from pathlib import Path

import mlflow
import pandas as pd

def _add_databricks_src_path() -> None:
    candidates = []
    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        notebook_root = Path(notebook_path).parent.parent
        candidates.extend([notebook_root / "src", notebook_root.parent / "src"])
    except Exception:
        pass

    cwd = Path.cwd()
    candidates.extend([cwd / "src", cwd / "databricks" / "src", cwd.parent / "databricks" / "src"])

    for candidate in candidates:
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))

_add_databricks_src_path()

try:
    from fraudshield_databricks import train_model_with_mlflow
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "No se pudo importar fraudshield_databricks. Verifica que databricks/src esté sincronizado en el repo de Databricks o que el notebook se ejecute dentro del workspace correcto."
    ) from error

ENGINEERED_DATA_PATH = "/Volumes/ml/fraudshield/data/credit_card_transactions_fe.parquet"
EXPERIMENT_NAME = "/Shared/FraudShield"
REGISTERED_MODEL_NAME = "FraudShield_LightGBM"

In [ ]:
df_spark = spark.read.parquet(ENGINEERED_DATA_PATH)
print(f"Spark rows: {df_spark.count():,}")
print(f"Spark columns: {len(df_spark.columns):,}")

df = df_spark.toPandas()
result = train_model_with_mlflow(
    df,
    experiment_name=EXPERIMENT_NAME,
    run_name="lightgbm_baseline_databricks",
    registered_model_name=REGISTERED_MODEL_NAME,
)

print(f'Run ID: {result.run_id}')
print(f'Model URI: {result.model_uri}')
print(f'Metrics: {result.metrics}')

In [ ]:
mlflow.set_experiment(EXPERIMENT_NAME)
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.pr_auc DESC", "start_time DESC"],
    max_results=10,
)

columns = [
    column
    for column in [
        "run_id",
        "metrics.pr_auc",
        "metrics.roc_auc",
        "metrics.precision",
        "metrics.recall",
        "status",
        "start_time",
    ]
    if column in runs.columns
]

display(runs[columns])

if not runs.empty:
    best_run_id = runs.iloc[0]["run_id"]
    print(f'Best run: {best_run_id}')
    print(f'Best model URI: runs:/{best_run_id}/model')

Revisa el experimento en la UI de MLflow para comparar PR-AUC, ROC-AUC, precision y recall.